In [4]:
import sys, os, yaml, pandas as pd, numpy as np, matplotlib
print("Python exe:", sys.executable)

CFG_PATH = "../src/config.yaml"
assert os.path.exists(CFG_PATH), f"Config not found at {CFG_PATH}"
CFG = yaml.safe_load(open(CFG_PATH, encoding="utf-8"))
print("Config paths:", CFG["paths"])

Python exe: C:\Users\eoinp\SmartMeterProject\.smp_env\Scripts\python.exe
Config paths: {'raw_dir': 'data/raw', 'clean_dir': 'data/clean', 'figs_dir': 'reports/figures', 'tables_dir': 'reports/tables', 'logs_dir': 'reports/logs'}


In [5]:
import sys
sys.executable


'C:\\Users\\eoinp\\SmartMeterProject\\.smp_env\\Scripts\\python.exe'

In [7]:
RAW_DIR   = os.path.join("..", CFG["paths"]["raw_dir"])
CLEAN_DIR = os.path.join("..", CFG["paths"]["clean_dir"])
FIGS_DIR  = os.path.join("..", CFG["paths"]["figs_dir"])
TABS_DIR  = os.path.join("..", CFG["paths"]["tables_dir"])
LOGS_DIR  = os.path.join("..", CFG["paths"]["logs_dir"])

for p in (CLEAN_DIR, FIGS_DIR, TABS_DIR, LOGS_DIR):
    os.makedirs(p, exist_ok=True)

print("RAW_DIR:", RAW_DIR)
print("Output dirs ok:", CLEAN_DIR, FIGS_DIR, TABS_DIR, LOGS_DIR)

RAW_DIR: ..\data/raw
Output dirs ok: ..\data/clean ..\reports/figures ..\reports/tables ..\reports/logs


In [8]:
import re

EXPECTED_TIMES = [f"{h:02d}:{m:02d}" for h in range(24) for m in (0,30)]
TIME_RE = re.compile(r"^\s*\d{1,2}:\d{2}\s*$")

def read_esb_wide(path):
    """Try multiple separators (autodetect → tab → comma → whitespace)."""
    def _read(**kw):
        return pd.read_csv(path, comment="#", dtype="string", **kw)

    for kw in [
        dict(sep=None, engine="python"),
        dict(sep="\t"),
        dict(sep=","),
        dict(delim_whitespace=True, engine="python")
    ]:
        try:
            df = _read(**kw)
            if df.shape[1] > 5:
                return df
        except Exception:
            pass
    raise ValueError(f"Unable to parse {path}")

def normalize_headers(df):
    df = df.copy()
    df.columns = (df.columns.astype(str)
                  .str.replace("\ufeff", "", regex=False)
                  .str.replace(r"\s+", " ", regex=True)
                  .str.strip())
    return df

def find_time_cols(cols):
    time_cols_raw = [c for c in cols if TIME_RE.match(str(c))]
    def to_hhmm(c):
        h,m = str(c).strip().split(":")
        return f"{int(h):02d}:{int(m):02d}"
    mapped = {c: to_hhmm(c) for c in time_cols_raw}
    mapped = {k:v for k,v in mapped.items() if v in EXPECTED_TIMES}
    return mapped

In [9]:
def detect_date_format(series, sample_size=50):
    """Guess a consistent date format like %Y-%m-%d or %d-%m-%Y."""
    vals = [str(x).strip() for x in series.dropna().astype(str).head(sample_size) if str(x).strip()]
    candidates = [
        ("%Y-%m-%d", r"^\d{4}-\d{2}-\d{2}$"),
        ("%d-%m-%Y", r"^\d{2}-\d{2}-\d{4}$"),
        ("%d/%m/%Y", r"^\d{2}/\d{2}/\d{4}$"),
        ("%m/%d/%Y", r"^\d{2}/\d{2}/\d{4}$"),
        ("%Y/%m/%d", r"^\d{4}/\d{2}/\d{2}$"),
        ("%d.%m.%Y", r"^\d{2}\.\d{2}\.\d{4}$")
    ]
    import re
    for fmt, pat in candidates:
        if all(re.match(pat, v) for v in vals):
            return fmt
    return None

In [10]:
def wide_to_long_strict(df_wide, dayfirst=False, verbose=True, file_hint=""):
    df_wide = normalize_headers(df_wide)
    cols = list(df_wide.columns)

    # Date column
    if "Date" in cols:
        date_col = "Date"
    else:
        cands = [c for c in cols if c.lower() == "date"]
        if not cands:
            raise ValueError(f"[{file_hint}] No 'Date' column found.")
        date_col = cands[0]

    # Detect date format
    detected_fmt = detect_date_format(df_wide[date_col])
    if verbose:
        print(f"[{file_hint}] Detected date format:", detected_fmt or f"(none; using dayfirst={dayfirst})")

    # Time columns
    time_map = find_time_cols(cols)
    print(f"[{file_hint}] Found {len(time_map)}/48 time columns.")

    df_renamed = df_wide.rename(columns=time_map)
    use_times = [t for t in EXPECTED_TIMES if t in df_renamed.columns]

    long = df_renamed.melt(id_vars=[date_col], value_vars=use_times,
                           var_name="hhmm", value_name="kWh")

    # Build timestamp with fixed date format if available
    date_str = long[date_col].astype("string").str.strip()
    if detected_fmt:
        date_only = pd.to_datetime(date_str, format=detected_fmt, errors="coerce")
        ts_text = date_only.dt.strftime("%Y-%m-%d") + " " + long["hhmm"]
        ts = pd.to_datetime(ts_text, format="%Y-%m-%d %H:%M", errors="coerce")
    else:
        ts = pd.to_datetime(date_str + " " + long["hhmm"], dayfirst=dayfirst, errors="coerce")

    long["timestamp"] = ts
    long["kWh"] = pd.to_numeric(long["kWh"], errors="coerce")
    long = long.dropna(subset=["timestamp"]).reset_index(drop=True)

    if verbose:
        ud = long["timestamp"].dt.date.nunique()
        print(f"[{file_hint}] Long rows: {len(long):,} over {ud} unique dates.")

    return long[["timestamp","kWh"]]

In [11]:
mode = CFG.get("run", {}).get("mode", "single")

if mode == "single":
    files_to_process = [CFG["run"]["input_file"]]
    LABELS = [CFG.get("run", {}).get("run_label", os.path.splitext(files_to_process[0])[0])]
else:
    files_to_process = CFG["run"]["input_files"]
    LABELS = [os.path.splitext(f)[0] for f in files_to_process]  # label = filename stem

runs = {}  # label -> long df

for fname, label in zip(files_to_process, LABELS):
    p = os.path.join(RAW_DIR, fname)
    print(f"\nProcessing {fname} as label='{label}' | Exists:", os.path.exists(p))
    assert os.path.exists(p), f"Missing raw file: {p}"

    dfw = read_esb_wide(p)
    print(f"[{fname}] Raw shape:", dfw.shape)

    dfl = wide_to_long_strict(dfw, dayfirst=CFG["data"]["dayfirst"], verbose=True, file_hint=fname)
    runs[label] = dfl

print("\nLoaded runs:", list(runs.keys()))
LABEL = list(runs.keys())[0]  # default: first run
raw = runs[LABEL].copy()

print("Selected LABEL:", LABEL)
print("Raw rows:", len(raw))
display(raw.head(6))



Processing 2024.csv as label='2024' | Exists: True
[2024.csv] Raw shape: (366, 49)
[2024.csv] Detected date format: %Y-%m-%d
[2024.csv] Found 48/48 time columns.
[2024.csv] Long rows: 17,568 over 366 unique dates.

Loaded runs: ['2024']
Selected LABEL: 2024
Raw rows: 17568


,timestamp,kWh
0,2024-01-01,0.137
1,2024-01-02,0.107
2,2024-01-03,0.163
3,2024-01-04,0.086
4,2024-01-05,0.105
5,2024-01-06,0.107


In [12]:
LOCAL_TZ = CFG["data"]["timezone_local"]

raw = raw.copy()
raw["timestamp"] = pd.to_datetime(raw["timestamp"], errors="coerce")
raw = raw.dropna(subset=["timestamp"]).sort_values("timestamp")

# Pick the FIRST occurrence for ambiguous fall-back times (DST),
# and shift-through the missing spring-forward hour.
raw["timestamp_local"] = raw["timestamp"].dt.tz_localize(
    LOCAL_TZ,
    ambiguous=True,             # <-- choose first (DST) occurrence
    nonexistent="shift_forward" # <-- handle spring-forward gap
)

raw["timestamp_utc"] = raw["timestamp_local"].dt.tz_convert("UTC")
raw = raw.drop(columns=["timestamp"]).dropna(subset=["timestamp_utc"])

raw[["timestamp_local","timestamp_utc","kWh"]].head(3)


,timestamp_local,timestamp_utc,kWh
0,2024-01-01 00:00:00+00:00,2024-01-01 00:00:00+00:00,0.137
366,2024-01-01 00:30:00+00:00,2024-01-01 00:30:00+00:00,0.145
732,2024-01-01 01:00:00+00:00,2024-01-01 01:00:00+00:00,0.9865


In [13]:
raw = raw.sort_values("timestamp_utc").drop_duplicates(subset=["timestamp_utc"], keep=CFG["policies"]["dedupe_keep"])

INTERVAL = f'{CFG["data"]["interval_minutes"]}min'
df = raw.set_index("timestamp_utc")[["kWh"]].sort_index()
s = df["kWh"].resample(INTERVAL).sum(min_count=1)
s = s.ffill(limit=CFG["policies"]["gap_fill_limit_intervals"])

out = pd.DataFrame({"kWh": s})
out["timestamp_local"] = out.index.tz_convert(LOCAL_TZ)
df = out
df.head(3)

,kWh,timestamp_local
timestamp_utc,,
2024-01-01 00:00:00+00:00,0.137,2024-01-01 00:00:00+00:00
2024-01-01 00:30:00+00:00,0.145,2024-01-01 00:30:00+00:00
2024-01-01 01:00:00+00:00,0.9865,2024-01-01 01:00:00+00:00


In [14]:
def daily_quality_flags_fair(dfx, daily_min=44):
    counts = dfx["kWh"].groupby(dfx["timestamp_local"].dt.date).count().astype(int)
    if len(counts) >= 2:
        counts = counts.iloc[1:-1]  # drop first/last (often partial in exports)
    ok = (counts >= daily_min) | (counts.isin([47,49]))  # allow DST transition
    return ~ok, counts

neg_mask = df["kWh"] < 0
df.loc[neg_mask, "kWh"] = np.nan
df["flag_negative"] = neg_mask

flags, counts = daily_quality_flags_fair(df, daily_min=CFG["policies"]["daily_completeness_min"])
print(f"Incomplete days (fair): {100*flags.mean():.1f}%")
print("Sample daily slot counts:", counts.head(10).tolist())

Incomplete days (fair): 0.0%
Sample daily slot counts: [48, 48, 48, 48, 48, 48, 48, 48, 48, 48]


In [15]:
# --- Cell 10 (fixed): helper cols & save clean CSV ---

# Use local wall-time without tz for calendar fields (prevents the warning)
local_naive = df["timestamp_local"].dt.tz_localize(None)

df["date"]       = local_naive.dt.date.astype("string")
df["month"]      = local_naive.dt.to_period("M").astype("string")   # e.g., '2023-09'
df["dow"]        = local_naive.dt.day_name()
df["is_weekend"] = local_naive.dt.weekday >= 5
# (Optional) a friendlier month label:
# df["month_label"] = local_naive.dt.strftime("%Y-%m")

clean_path = os.path.join(CLEAN_DIR, f"clean_{LABEL}_30min.csv")
df.to_csv(clean_path, index_label="timestamp_utc", encoding="utf-8")
print("Saved:", clean_path)

df.head(3)


Saved: ..\data/clean\clean_2024_30min.csv


,kWh,timestamp_local,flag_negative,date,month,dow,is_weekend
timestamp_utc,,,,,,,
2024-01-01 00:00:00+00:00,0.137,2024-01-01 00:00:00+00:00,False,2024-01-01,2024-01,Monday,False
2024-01-01 00:30:00+00:00,0.145,2024-01-01 00:30:00+00:00,False,2024-01-01,2024-01,Monday,False
2024-01-01 01:00:00+00:00,0.9865,2024-01-01 01:00:00+00:00,False,2024-01-01,2024-01,Monday,False


In [16]:
# Recreate/ensure flags & counts exist (safe if already defined)
def daily_quality_flags_fair(dfx, daily_min=44):
    counts = dfx["kWh"].groupby(dfx["timestamp_local"].dt.date).count().astype(int)
    if len(counts) >= 2:
        counts = counts.iloc[1:-1]  # drop first/last partial export days
    ok = (counts >= daily_min) | (counts.isin([47,49]))  # DST tolerance
    return ~ok, counts

flags, counts = daily_quality_flags_fair(df, daily_min=CFG["policies"]["daily_completeness_min"])
valid_local_days = set(counts.index[~flags])

# Filter to valid days + non-NaN kWh
df["local_day"] = df["timestamp_local"].dt.date
dfv = df[df["local_day"].isin(valid_local_days)].dropna(subset=["kWh"]).copy()

# 48 half-hour slot labels in local time
slots = [f"{h:02d}:{m:02d}" for h in range(24) for m in (0,30)]
dfv["slot"] = dfv["timestamp_local"].dt.strftime("%H:%M")

print("Valid days:", len(valid_local_days))
dfv.head(3)

Valid days: 364


,kWh,timestamp_local,flag_negative,date,month,dow,is_weekend,local_day,slot
timestamp_utc,,,,,,,,,
2024-01-02 00:00:00+00:00,0.107,2024-01-02 00:00:00+00:00,False,2024-01-02,2024-01,Tuesday,False,2024-01-02,00:00
2024-01-02 00:30:00+00:00,0.079,2024-01-02 00:30:00+00:00,False,2024-01-02,2024-01,Tuesday,False,2024-01-02,00:30
2024-01-02 01:00:00+00:00,0.353,2024-01-02 01:00:00+00:00,False,2024-01-02,2024-01,Tuesday,False,2024-01-02,01:00


In [31]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

typical = dfv.groupby("slot")["kWh"].mean().reindex(slots)

# Save table
typ_csv = os.path.join(TABS_DIR, f"typical_day_{LABEL}.csv")
typical.to_csv(typ_csv, encoding="utf-8")
print("Saved:", typ_csv)

# -----------------------------
# Colour-code by TOU time bands
# Default bands (common example / Electric Ireland-style):
# Off-peak: 23:00–08:00
# Peak:     17:00–19:00
# Mid-peak: everything else
# -----------------------------
def classify_tou(slot_str: str) -> str:
    # slot_str is "HH:MM"
    hh, mm = map(int, slot_str.split(":"))
    minutes = hh * 60 + mm

    off_peak = (minutes >= 23 * 60) or (minutes < 8 * 60)         # 23:00–08:00
    on_peak  = (minutes >= 17 * 60) and (minutes < 19 * 60)       # 17:00–19:00

    if on_peak:
        return "On Peak"
    if off_peak:
        return "Off Peak"
    return "Mid Peak"

tou = pd.Series([classify_tou(t) for t in typical.index], index=typical.index)

# You asked for colour-coding, so we set explicit colours.
# (If you want different colours later, just change these 3.)
colour_map = {
    "Off Peak": "#4CAF50",   # green
    "Mid Peak": "#FFC107",   # amber
    "On Peak":  "#F44336",   # red
}
bar_colours = [colour_map[c] for c in tou.values]

# Plot
plt.figure(figsize=(16, 5))

x = np.arange(len(typical.index))
plt.bar(x, typical.values, color=bar_colours, width=0.85)

plt.title("Typical Day Profile (Average kWh per 30-min)")
plt.xlabel("Time of Day (Europe/Dublin)")
plt.ylabel("kWh / 30-min")

# Show ticks every 2 hours (every 4 half-hours)
tick_step = 4
tick_idx = list(x[::tick_step])
tick_labels = [slots[i] for i in tick_idx]

# Add an end-of-day marker label (24:00)
tick_idx.append(len(slots))
tick_labels.append("24:00")
plt.xticks(tick_idx, tick_labels)

plt.xlim(-0.5, len(slots))
plt.grid(True, axis="y")
plt.tight_layout()

# Manual legend (ensures legend shows categories even though colours are per-bar)
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=colour_map[k], label=k) for k in ["Off Peak", "Mid Peak", "On Peak"]]
plt.legend(handles=legend_handles, loc="upper right")

outp = os.path.join(FIGS_DIR, f"typical_day_{LABEL}.png")
plt.savefig(outp, dpi=CFG["policies"]["export_dpi"])
plt.close()
print("Saved:", outp)



Saved: ..\reports/tables\typical_day_2024.csv
Saved: ..\reports/figures\typical_day_2024.png


In [30]:
wk = dfv[~dfv["is_weekend"]].groupby("slot")["kWh"].mean().reindex(slots)
we = dfv[ dfv["is_weekend"]].groupby("slot")["kWh"].mean().reindex(slots)
wk_we = pd.DataFrame({"Weekday": wk, "Weekend": we})
wk_we_path = os.path.join(TABS_DIR, f"weekday_weekend_profile_{LABEL}.csv")
wk_we.to_csv(wk_we_path, encoding="utf-8")
print("Saved:", wk_we_path)

plt.figure(figsize=(14, 5))

# x positions for the 48 half-hour slots
x = np.arange(len(wk_we.index))

plt.plot(x, wk_we["Weekday"].values, label="Weekday")
plt.plot(x, wk_we["Weekend"].values, label="Weekend")

plt.title("Weekday vs Weekend (Average kWh every 30-min)")
plt.xlabel("Time of Day")
plt.ylabel("kWh / 30-min")

# Show ticks every 2 hours (every 4 half-hour slots)
tick_step = 4
tick_idx = list(x[::tick_step])
tick_labels = [slots[i] for i in tick_idx]

# Add visual end-of-day marker
tick_idx.append(len(slots))      # 48
tick_labels.append("24:00")

plt.xticks(tick_idx, tick_labels, rotation=0)
plt.xlim(-0.5, len(slots))

plt.grid(True, axis="y")
plt.legend()
plt.tight_layout()

outp = os.path.join(FIGS_DIR, f"weekday_weekend_overlay_{LABEL}.png")
plt.savefig(outp, dpi=CFG["policies"]["export_dpi"])
plt.close()
print("Saved:", outp)


Saved: ..\reports/tables\weekday_weekend_profile_2024.csv
Saved: ..\reports/figures\weekday_weekend_overlay_2024.png


In [24]:
# --- Cell 15 (updated): Monthly totals + nicer month labels + seasonal split ---

import matplotlib.pyplot as plt
import calendar

# Monthly totals (kWh)
monthly = df.groupby(df["timestamp_local"].dt.to_period("M"))["kWh"].sum().to_timestamp()

# Save table
monthly_path = os.path.join(TABS_DIR, f"monthly_totals_{LABEL}.csv")
monthly.to_csv(monthly_path, encoding="utf-8")
print("Saved:", monthly_path)

# Build month tick labels like: Jan Feb Mar ...
month_nums = monthly.index.month
month_labels = [calendar.month_abbr[m] for m in month_nums]  # Jan, Feb, ...

# --- Plot using categorical positions ---
x = np.arange(len(monthly))  # 0..11

plt.figure()
plt.bar(x, monthly.values, width=0.6)

plt.title("Monthly Energy Use (kWh)")
plt.xlabel("Months")
plt.ylabel("kWh")

plt.xticks(x, month_labels)   # Jan Feb Mar ...
plt.grid(True, axis="y")
plt.tight_layout()

outp = os.path.join(FIGS_DIR, f"monthly_totals_{LABEL}.png")
plt.savefig(outp, dpi=CFG["policies"]["export_dpi"])
plt.close()
print("Saved:", outp)

# Optional: simple heating season vs non-heating (Nov–Mar vs Apr–Oct)
m = df["timestamp_local"].dt.month
is_heating = m.isin([11, 12, 1, 2, 3])
seasonal = pd.Series({
    "Heating (Nov–Mar)": df.loc[is_heating, "kWh"].sum(),
    "Non-heating (Apr–Oct)": df.loc[~is_heating, "kWh"].sum()
})

seasonal_path = os.path.join(TABS_DIR, f"seasonal_totals_{LABEL}.csv")
seasonal.to_csv(seasonal_path, encoding="utf-8")
print("Saved:", seasonal_path)


Saved: ..\reports/tables\monthly_totals_2024.csv
Saved: ..\reports/figures\monthly_totals_2024.png
Saved: ..\reports/tables\seasonal_totals_2024.csv


C:\Users\eoinp\AppData\Local\Temp\ipykernel_10352\3921259542.py:7: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  monthly = df.groupby(df["timestamp_local"].dt.to_period("M"))["kWh"].sum().to_timestamp()


In [20]:
# Annual totals by local calendar year
annual = df.groupby(df["timestamp_local"].dt.year)["kWh"].sum()

# Average daily kWh per year (use valid days only)
avg_daily_per_year = {}
for yr, sub in df.groupby(df["timestamp_local"].dt.year):
    days = sub["timestamp_local"].dt.date
    avg_daily_per_year[yr] = sub["kWh"].sum() / pd.Series(days).nunique()

# Peak time slot (Typical Day max)
peak_slot = typical.idxmax()
peak_slot_kwh = float(typical.max())

# Base load estimate: median of the *lowest* 20% of slots in the Typical Day
n20 = max(1, int(0.2 * len(typical)))
base_load = float(typical.nsmallest(n20).median())

# Weekend share of total energy
weekend_total = df.loc[df["is_weekend"], "kWh"].sum()
overall_total = df["kWh"].sum()
weekend_share_pct = 100.0 * weekend_total / overall_total if overall_total else 0.0

# Assemble KPI table
kpi_rows = []
for yr in sorted(annual.index):
    kpi_rows.append({
        "year": int(yr),
        "annual_kWh": float(annual.loc[yr]),
        "avg_daily_kWh": float(avg_daily_per_year.get(yr, np.nan)),
        "peak_slot_local": peak_slot,
        "peak_slot_kWh": round(peak_slot_kwh, 4),
        "base_load_kWh_per_30min": round(base_load, 4),
        "weekend_share_percent": round(weekend_share_pct, 2),
    })

kpis = pd.DataFrame(kpi_rows)
kpi_path = os.path.join(TABS_DIR, f"kpis_{LABEL}.csv")
kpis.to_csv(kpi_path, index=False, encoding="utf-8")
print("Saved:", kpi_path)
kpis

Saved: ..\reports/tables\kpis_2024.csv


,year,annual_kWh,avg_daily_kWh,peak_slot_local,peak_slot_kWh,base_load_kWh_per_30min,weekend_share_percent
0,2024,5066.4385,13.842728,18:30,0.4192,0.2021,28.18


In [21]:
from datetime import datetime

md = f"""# C2 – Usage Profiling Summary

**Run timestamp:** {datetime.now().isoformat(timespec='seconds')}
**Data window:** {df['timestamp_local'].min().date()} → {df['timestamp_local'].max().date()}
**Timezone policy:** Process UTC, present Europe/Dublin.
**Cadence:** 30-minute intervals (48 slots/day); tiny gaps ffilled (limit={CFG["policies"]["gap_fill_limit_intervals"]}).

## Artifacts
- Clean canonical table: `data/clean/clean_{LABEL}_30min.csv`
- Typical day CSV/PNG: `reports/tables/typical_day_{LABEL}.csv`, `reports/figures/typical_day_{LABEL}.png`
- Weekday vs Weekend CSV/PNG: `reports/tables/weekday_weekend_profile_{LABEL}.csv`, `reports/figures/weekday_weekend_overlay_{LABEL}.png`
- Monthly totals CSV/PNG: `reports/tables/monthly_totals_{LABEL}.csv`, `reports/figures/monthly_totals_{LABEL}.png`
- Seasonal totals CSV: `reports/tables/seasonal_totals_{LABEL}.csv` (optional)
- KPI table: `reports/tables/kpis_{LABEL}.csv`

## Notes / Assumptions
- DST handled with `ambiguous=True` (choose first fall-back hour), spring gap shifted forward.
- Incomplete days (fair): {100*flags.mean():.1f}% (first/last export day excluded; DST days allowed at 47/49 slots).
- Negative readings set to NaN and flagged (`flag_negative`).
- PII removed/upstream (no MPRN saved).

## Next (C3)
Use `data/clean/clean_2023_2025_30min.csv` with columns:
`timestamp_utc` (index), `timestamp_local`, `kWh`, `date`, `month`, `dow`, `is_weekend`, `flag_negative`.
"""

rp = os.path.join(LOGS_DIR, f"report_profile_summary_{LABEL}.md")
with open(rp, "w", encoding="utf-8") as f:
    f.write(md)
print("Saved:", rp)


Saved: ..\reports/logs\report_profile_summary_2024.md


In [22]:
# Build day completeness (0..48) in local time
day_counts = df["kWh"].groupby(df["timestamp_local"].dt.date).count().astype(int)

# Put into a 7xN matrix by weekday for a simple calendar-like plot
# (Minimal matplotlib version; no seaborn.)
import math
import matplotlib.pyplot as plt
dates = pd.to_datetime(pd.Series(list(day_counts.index)))
vals = day_counts.reindex(dates.dt.date).values

dow = dates.dt.weekday.values         # 0=Mon
week = ((dates - dates.min()).dt.days // 7).values
wmax = week.max()+1
grid = np.full((7, wmax), np.nan)
for d, w, v in zip(dow, week, vals):
    grid[d, w] = v

plt.figure()
plt.imshow(grid, aspect="auto", interpolation="nearest")
plt.title("Daily completeness (slots per day, 0–48)")
plt.ylabel("Day of week (0=Mon)")
plt.xlabel("Week index from start")
plt.colorbar(label="slots (0–48)")
plt.tight_layout()
outp = os.path.join(FIGS_DIR, f"completeness_calendar_{LABEL}.png")
plt.savefig(outp, dpi=CFG["policies"]["export_dpi"])
plt.close()
print("Saved:", outp)


Saved: ..\reports/figures\completeness_calendar_2024.png
